# Energy Simulation: Baseline (weight only) vs Smart (weight + area)

Models a synthetic stream of hall calls and compares the two control strategies on:
- # accepted vs bypassed stops
- estimated total energy
- proxy AWT impact

In [ ]:
import sys, os, random
ROOT = os.path.abspath('..')
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import numpy as np
import matplotlib.pyplot as plt
from src.utils.config_loader import ElevatorConfig
from src.energy.consumption import EnergyParams, StartProfile, estimate_session_energy, estimate_stop_energy

In [ ]:
cfg = ElevatorConfig.from_yaml('../configs/default.yaml').to_dict()
p = EnergyParams.from_config(cfg)
p

## Single-stop energy as a function of load

In [ ]:
loads = np.linspace(0, p.rated_load_kg, 30)
ups = [estimate_stop_energy(StartProfile(l, 1, True), p)['total_kj'] for l in loads]
downs = [estimate_stop_energy(StartProfile(l, 1, False), p)['total_kj'] for l in loads]

plt.figure(figsize=(8, 4.5))
plt.plot(loads, ups, label='Going up')
plt.plot(loads, downs, label='Going down')
plt.xlabel('Load (kg)'); plt.ylabel('Stop energy (kJ)')
plt.title('Single-stop energy vs load — 1 floor traversal')
plt.grid(alpha=0.3); plt.legend(); plt.tight_layout(); plt.show()

## Daily simulation: 200 hall calls — Smart strategy bypasses 25%

In [ ]:
rng = random.Random(42)
calls = []
for _ in range(200):
    load = rng.uniform(50, p.rated_load_kg)
    floors = rng.randint(1, 5)
    direction = rng.random() < 0.5
    calls.append(StartProfile(load, floors, direction))

# BASELINE: weight-only bypass at 80% of rated load
baseline_accept = [c for c in calls if c.load_kg / p.rated_load_kg < 0.80]
baseline_bypass = [c for c in calls if c not in baseline_accept]

# SMART: assume vision detects spatial fullness for additional 18% of calls
extra_bypass_idx = rng.sample(range(len(baseline_accept)), int(0.18 * len(baseline_accept)))
smart_accept = [c for i, c in enumerate(baseline_accept) if i not in extra_bypass_idx]
smart_bypass = baseline_bypass + [baseline_accept[i] for i in extra_bypass_idx]

stationary_seconds = 8 * 3600  # idle ~8 h/day
e_base = estimate_session_energy(baseline_accept, baseline_bypass, stationary_seconds, p)
e_smart = estimate_session_energy(smart_accept, smart_bypass, stationary_seconds, p)

print(f'BASELINE: stops={e_base.starts:>3d}  bypass={e_base.bypassed:>3d}  '
      f'total={e_base.total_kwh:.3f} kWh')
print(f'SMART   : stops={e_smart.starts:>3d}  bypass={e_smart.bypassed:>3d}  '
      f'total={e_smart.total_kwh:.3f} kWh')
savings_pct = (e_base.total_j - e_smart.total_j) / e_base.total_j * 100
print(f'\nSavings : {savings_pct:.1f}%  ({(e_base.total_j - e_smart.total_j) / 3.6e6:.3f} kWh)')

In [ ]:
labels = ['Baseline\n(weight only)', 'Smart\n(weight + area)']
components = ['running', 'aux_motion', 'doors', 'stationary']
vals = np.array([
    [e.total_running_j, e.total_aux_j, e.total_doors_j, e.total_stationary_j]
    for e in (e_base, e_smart)
]) / 1000  # kJ

fig, ax = plt.subplots(figsize=(7, 4.5))
bottom = np.zeros(2)
for i, name in enumerate(components):
    ax.bar(labels, vals[:, i], bottom=bottom, label=name)
    bottom += vals[:, i]
ax.set_ylabel('Energy (kJ)')
ax.set_title('Daily energy breakdown (synthetic 200 hall calls)')
ax.legend(); ax.grid(alpha=0.3, axis='y')
plt.tight_layout(); plt.show()

## Annual extrapolation

If a building runs ~250 traffic-days/year:

In [ ]:
annual_base = e_base.total_kwh * 250
annual_smart = e_smart.total_kwh * 250
print(f'Annual baseline: {annual_base:.1f} kWh')
print(f'Annual smart   : {annual_smart:.1f} kWh')
print(f'Annual saving  : {annual_base - annual_smart:.1f} kWh ({savings_pct:.1f}%)')